In [1]:
import os
os.chdir('/workspace/db7b8b85-44c7-4bc8-874a-718dd84baeab')
print(os.listdir('.'))


['lchi_zeros_5000_dps50.npy', 'ldh_zeros_partial_dps50.npy', 'ldelta_zeros_2000_dps50.npy', 'ldh_def.py', 'memory', 'omega_class_moments.csv', '.config', '.kernel_llm_logs_1.txt', '.prompts']


In [2]:
import pandas as pd
df = pd.read_csv('omega_class_moments.csv')
print(df)
print(df.columns.tolist())


 L_function N_terms M_total M_0 M_1 M_2 \
0 zeta 10000 14049.610398 500.0 1965.747008 6483.537674 
1 zeta 50000 52555.776641 500.0 3208.301641 21414.200073 
2 zeta 100000 98775.232940 500.0 4480.038022 37384.580762 
3 L(Δ) 10000 2161.019922 500.0 976.303859 566.192078 
4 L(Δ) 50000 2348.222814 500.0 1033.094242 641.389207 
5 L(Δ) 100000 2428.171224 500.0 1050.861296 676.608911 
6 L_DH 10000 2025.689869 500.0 618.478717 717.769427 
7 L_DH 50000 2186.052439 500.0 639.066491 784.006896 
8 L_DH 100000 2238.888403 500.0 644.808316 800.888886 

 M_3 M_4 M_geq5 frac_0 frac_1 frac_2 \
0 4759.568118 337.922395 2.835203 0.035588 0.139915 0.461475 
1 24096.365734 3301.861669 35.047524 0.009514 0.061046 0.407457 
2 48011.972434 8267.691145 130.950577 0.005062 0.045356 0.378481 
3 113.409894 5.078208 0.035883 0.231372 0.451779 0.262002 
4 159.091605 14.413875 0.233885 0.212927 0.439947 0.273138 
5 181.308525 18.942436 0.450056 0.205916 0.432779 0.278650 
6 168.571130 20.716949 0.153646 0.246829 0.3

In [3]:
import numpy as np
# Plan check: For N=10^6 and many t-grid points, doing outer(t_grid, log_n) all at once is huge.
# Need approach: process via chunking. 
# Memory: N=10^6 with T=50000 t-points => 5e10 floats; impossible.
# Better: process per-k subset by chunks of n; accumulate S_k(t) as complex sum.

# But t-grid size: dt should be < 2*pi/log(N) ~ 2*pi/13.8 ~ 0.46. So dt=0.05-0.1 sufficient.
# T=500/dt. dt=0.05 -> T=10000. Or dt=0.025 -> 20000.
# For r38 csv, let's verify by recomputing the existing N=1e4 zeta value with various T.

# Best plan: use FFT-like approach? Actually direct evaluation: S_k(t) = sum_n amp_n exp(-i*t*log_n)
# Process n in chunks of e.g. 10000 to keep matrix size T*chunk small.

# Memory per chunk: T=10000, chunk=20000 -> 2e8 complex = 3.2GB; too much.
# Use chunk=5000 -> 5e7 complex = 800MB; doable. Or use float32 phase + complex64.

# Actually a cleaner approach: process per omega-class. Compute S_k(t) accumulating with vectorized chunks.
# Let's first reproduce N=10^5 zeta result for sanity.

from sympy import sieve
import time

def omega_sieve(N):
 """smallest prime factor sieve -> omega(n) = #distinct primes."""
 spf = np.zeros(N+1, dtype=np.int32)
 for p in range(2, N+1):
 if spf[p] == 0:
 for m in range(p, N+1, p):
 if spf[m] == 0:
 spf[m] = p
 omega = np.zeros(N+1, dtype=np.int8)
 for n in range(2, N+1):
 m = n
 last = 0
 c = 0
 while m > 1:
 p = spf[m]
 if p != last:
 c += 1
 last = p
 m //= p
 omega[n] = c
 return omega

t0=time.time()
om = omega_sieve(10000)
print("done", time.time()-t0, om[:20])


done 0.012825250625610352 [0 0 1 1 1 1 2 1 1 1 2 1 2 1 2 2 1 1 2 1]


In [4]:
# Faster omega via vectorized approach for N=10^6
def omega_vec(N):
 omega = np.zeros(N+1, dtype=np.int8)
 is_composite = np.zeros(N+1, dtype=bool)
 for p in range(2, N+1):
 if not is_composite[p]:
 # mark multiples and increment omega
 idx = np.arange(p, N+1, p)
 omega[idx] += 1
 is_composite[idx[1:]] = True # skip p itself; but we need to also mark prime
 is_composite[p] = False # p itself not composite
 return omega

t0=time.time()
om = omega_vec(100000)
print("ok", time.time()-t0)
print(om[:20])
# Compare with my reference
om_ref = omega_sieve(100000)
print((om==om_ref).all())


ok 0.03778886795043945
[0 0 1 1 1 1 2 1 1 1 2 1 2 1 2 2 1 1 2 1]
True


In [5]:
t0=time.time()
om_1m = omega_vec(1_000_000)
print("omega(10^6) done in", time.time()-t0, "s")
import collections
print(collections.Counter(om_1m[1:].tolist()))


omega(10^6) done in 0.33083558082580566 s
Counter({3: 379720, 2: 288726, 4: 208034, 1: 78734, 5: 42492, 6: 2285, 7: 8, 0: 1})


In [6]:
# Now coefficients
# For L(Δ): a_n = tau(n) / n^(11/2). Use cypari2
import cypari2
pari = cypari2.Pari()

t0=time.time()
# PARI: ramanujantau(n) - exists
# Use vectorized: gp.eval("vector(N, n, ramanujantau(n))") but slow
# Better use the multiplicative structure: tau is multiplicative, tau(p^k) recurrence:
# tau(p^(k+1)) = tau(p)*tau(p^k) - p^11 * tau(p^(k-1))
# We need tau(p) for primes p <= N, then multiplicativity.

N = 1_000_000
# Get primes up to N
import sympy
print("getting primes...")
primes_list = list(sympy.primerange(2, N+1))
print(len(primes_list), "primes; time", time.time()-t0)


ModuleNotFoundError: No module named 'cypari2'

In [7]:
import subprocess
print(subprocess.run(["pip","install","cypari2","-q"], capture_output=True, text=True).stdout[-500:])
print(subprocess.run(["pip","install","cypari2","-q"], capture_output=True, text=True).stderr[-500:])


In [8]:
import cypari2
pari = cypari2.Pari()
print(pari("ramanujantau(10)"))


-115920


In [9]:
# Compute tau values via multiplicative property to be fast.
# Step 1: tau(p) for each prime p <= N (call PARI per prime).
# But for 78498 primes (up to 1M), one-by-one PARI calls may be slow.
# PARI has 'mfcoefs' for modular forms: f = mfDelta; mfcoefs(f, N) returns first N+1 coefficients.

import time
t0=time.time()
# PARI: mfDelta = mfinit([1,12], 1)... Actually `mfDelta` corresponds to the unique cusp form.
# Use: D = mfinit([1,12,1],1); f = mfeigenbasis(D)[1]; mfcoefs(f, N) ; or simpler: mfeigensearch / mfDelta = lift of weight 12 level 1.

# Simpler: use newform = mfinit([1,12]); then ramanujantau already exists.

# Try direct: vec = vector(N, n, ramanujantau(n)) - might be slow. Let's test on 10000.
v = pari(f"vector(10000, n, ramanujantau(n))")
print("time", time.time()-t0)
print(len(v), v[0], v[1], v[9])


time 0.04033660888671875
10000 1 -24 -115920


In [10]:
# very fast! Try 1e6
t0=time.time()
v_tau = pari(f"vector(1000000, n, ramanujantau(n))")
print("time", time.time()-t0)
print(len(v_tau))


 *** vector: Warning: increasing stack size to 8003584.


PariError: ramanujantau: the PARI stack overflows (current size: 8003584; maximum size: 8003584)
You can use pari.allocatemem() to change the stack size and try again

In [11]:
pari.allocatemem(2**30)
t0=time.time()
v_tau = pari(f"vector(1000000, n, ramanujantau(n))")
print("time", time.time()-t0, len(v_tau))


TimeoutError: Code execution timed out after 900.0 seconds

In [12]:
# too slow. Use multiplicative property + mfcoefs.
# In PARI: 
# mf = mfinit([1,12,1],1); F=mfeigenbasis(mf)[1]; coefs = mfcoefs(F, N);
# But might not include leading term. Let's try small.
import time
t0=time.time()
res = pari("mf=mfinit([1,12],1); F=mfeigenbasis(mf)[1]; mfcoefs(F,20)")
print("time", time.time()-t0)
print(res)


TimeoutError: Code execution timed out after 900.0 seconds

In [13]:
# PARI got stuck on prior operation. Reset.
import cypari2
pari = cypari2.Pari()
pari.allocatemem(2**30)
import time
t0=time.time()
res = pari("mf=mfinit([1,12],1); F=mfeigenbasis(mf)[1]; mfcoefs(F,20)")
print("time", time.time()-t0)
print(res)


TimeoutError: Code execution timed out after 900.0 seconds

In [ ]:
# KERNEL_RESET

In [1]:
import os, time
os.chdir('/workspace/db7b8b85-44c7-4bc8-874a-718dd84baeab')
import numpy as np
import pandas as pd

# Compute tau(n) via multiplicative property.
# Use a sieve: smallest prime factor (spf) + n = p^k * m with gcd(p,m)=1.
# tau(p^k) recurrence: tau(p^(k+1)) = tau(p)*tau(p^k) - p^11 * tau(p^(k-1))
# Need tau(p) for each prime p <= N.

# Get tau(p) by calling PARI per prime — but only ~78498 primes. Try smaller approach first:
# Actually for our purpose we ONLY need a_n = tau(n)/n^(11/2) — these are float64 numbers in [-2,2] ish.
# So work in float64 from the start (avoid huge integers).

N = 1_000_000

# Build smallest prime factor sieve
t0=time.time()
spf = np.zeros(N+1, dtype=np.int32)
for p in range(2, int(N**0.5)+1):
 if spf[p] == 0:
 spf[p*p::p] = np.where(spf[p*p::p]==0, p, spf[p*p::p])
# fill primes themselves
for p in range(2, N+1):
 if spf[p]==0:
 spf[p] = p
print("spf done", time.time()-t0)


spf done 0.1582331657409668


In [2]:
# Faster spf
t0=time.time()
spf = np.zeros(N+1, dtype=np.int32)
for p in range(2, N+1):
 if spf[p] == 0:
 spf[p] = p
 if p*p <= N:
 mask = spf[p*p::p] == 0
 indices = np.arange(p*p, N+1, p)[mask]
 spf[indices] = p
print("spf done", time.time()-t0)
print(spf[:20])


spf done 0.16078710556030273
[ 0 0 2 3 2 5 2 7 2 3 2 11 2 13 2 3 2 17 2 19]


In [3]:
# Get list of primes
primes = np.where(spf == np.arange(N+1))[0]
primes = primes[primes>=2]
print(len(primes), "primes up to", N)

# Compute omega(n)
t0=time.time()
omega = np.zeros(N+1, dtype=np.int8)
is_comp = np.zeros(N+1, dtype=bool)
for p in primes:
 idx = np.arange(p, N+1, p)
 omega[idx] += 1
print("omega done", time.time()-t0)
import collections
print(collections.Counter(omega[1:].tolist()))


78498 primes up to 1000000


omega done 0.2471017837524414
Counter({3: 379720, 2: 288726, 4: 208034, 1: 78734, 5: 42492, 6: 2285, 7: 8, 0: 1})


In [4]:
# Now get tau(p) for each prime p using PARI ramanujantau (one at a time, but only for primes)
import cypari2
pari = cypari2.Pari()
pari.allocatemem(2**29)

# Batch: do them in chunks; ramanujantau on single int is fast
# 78498 primes — let's time on 1000 primes first
t0=time.time()
tau_test = [int(pari(f"ramanujantau({p})")) for p in primes[:1000].tolist()]
print("1000 primes time:", time.time()-t0)
# extrapolate


PARI stack size set to 536870912 bytes, maximum size set to 536870912
1000 primes time: 0.03595161437988281


In [5]:
# Fast enough. Do all primes.
t0=time.time()
tau_p_list = [int(pari(f"ramanujantau({int(p)})")) for p in primes]
print("all primes tau time:", time.time()-t0)
tau_p = dict(zip(primes.tolist(), tau_p_list))
print("samples:", tau_p[2], tau_p[3], tau_p[5])


TimeoutError: Code execution timed out after 34.0 seconds